# Profiling Python code

**Goal:** find all duplicate titles in a list of 10,000 movies, measure where the time goes with `cProfile`, then make it fast.

The slow version follows the example by Sebastiaan Mathôt that was used in class; the profiling run and the fast version are my additions.

In [1]:
import cProfile, io, pstats, time
from collections import Counter

def read_movies(src='movies.txt'):
    """Return the movie titles in a text file as a list."""
    with open(src, encoding='utf-8') as f:
        return f.read().splitlines()

movies = read_movies()
len(movies)

10000

## 1. Slow version: compare every title with every other title

In [2]:
def is_duplicate(item, collection):
    for movie in collection:
        if movie.lower() == item.lower():
            return True
    return False

def find_duplicates_slow(movie_list):
    movie_list = list(movie_list)
    duplicates = []
    while movie_list:
        movie = movie_list.pop()
        if is_duplicate(movie, movie_list):
            duplicates.append(movie)
    return duplicates

## 2. Profile it

In [3]:
def profile(func):
    """Decorator: run func under cProfile and print the 8 most expensive calls."""
    def inner(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        pstats.Stats(pr, stream=s).sort_stats('cumulative').print_stats(8)
        print(s.getvalue())
        return result
    return inner

slow = profile(find_duplicates_slow)(movies)

         98214671 function calls (98214670 primitive calls) in 32.763 seconds

   Ordered by: cumulative time
   List reduced from 87 to 8 due to restriction <8>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      9/8    0.000    0.000   32.719    4.090 /usr/local/lib/python3.13/asyncio/base_events.py:1981(_run_once)
    10000   17.073    0.002   32.629    0.003 /tmp/ipykernel_8950/1639686569.py:1(is_duplicate)
        7    0.013    0.002   28.943    4.135 /usr/local/lib/python3.13/selectors.py:435(select)
 98193766   15.611    0.000   15.611    0.000 {method 'lower' of 'str' objects}
        1    0.003    0.003    3.051    3.051 /tmp/ipykernel_8950/1639686569.py:7(find_duplicates_slow)
       31    0.044    0.001    0.082    0.003 {built-in method time.sleep}
        8    0.000    0.000    0.043    0.005 /usr/local/lib/python3.13/asyncio/events.py:87(_run)
        8    0.000    0.000    0.043    0.005 {method 'run' of '_contextvars.Context' objects}





Almost all the time goes into `is_duplicate` and the almost 100 million `str.lower()` calls inside it: every title is lower-cased again for every comparison, and each title is compared with all remaining titles (quadratic work).

## 3. Fast version: lower-case once and count

In [4]:
def find_duplicates_fast(movie_list):
    counts = Counter(m.lower() for m in movie_list)
    return [title for title, n in counts.items() if n > 1]

for func in (find_duplicates_slow, find_duplicates_fast):
    t0 = time.perf_counter()
    result = func(movies)
    print(f'{func.__name__:22s} {time.perf_counter() - t0:8.3f} s   {len(set(m.lower() for m in result))} duplicate titles')

find_duplicates_slow      5.354 s   263 duplicate titles
find_duplicates_fast      0.002 s   263 duplicate titles


In [5]:
# both versions find the same titles
assert {m.lower() for m in slow} == set(find_duplicates_fast(movies))
sorted(find_duplicates_fast(movies))[:10]

['11:14 (2003)',
 '12 years a slave (2013)',
 '3 ninjas (1992)',
 '7 faces of dr. lao (1964)',
 '9 (2009)',
 'a christmas story live! (2017)',
 'adventures of ichabod and mr. toad, the (1949)',
 'alice (2009)',
 'alice (neco z alenky) (1988)',
 'alien: covenant (2017)']